In [ ]:
from notebooks.features.feature_extraction import load_all_features

features = ['PFRED_PLS', 'PFRED_SVM', 'OW_Overall', 'OW_Tm', 'OW_Intra_Oligo', 'OW_Duplex', 'sfold_accessibility', 'miranda_score', 'miranda_energy']
all_features = load_all_features(version='oligo', load_competition=True)

In [ ]:
from notebooks.consts import OLIGO_CSV_PROCESSED
import pandas as pd

oligo_data = pd.read_csv(OLIGO_CSV_PROCESSED)

In [ ]:
oligo_data = pd.merge(oligo_data, all_features, on='index_oligo')

In [ ]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression

# 1. Define the cohort column
oligo_data['cohort'] = oligo_data[CANONICAL_GENE] + "_" + oligo_data[CELL_LINE]

def get_comprehensive_stats(df, feature_cols, target_col=INHIBITION):
    results = []

    for col in feature_cols:
        spearman_list = []
        mi_list = []

        # Iterate through cohorts
        for cohort, group in df.groupby('cohort'):
            # Data cleaning per cohort
            valid_data = group[[col, target_col]].dropna()

            # Filtering out small cohorts to avoid noise
            if len(valid_data) < 5:
                continue

            x = valid_data[[col]]
            y = valid_data[target_col]

            # Spearman Correlation
            spearman_val = valid_data[col].corr(valid_data[target_col], method='spearman')
            spearman_list.append(spearman_val)

            # Mutual Information
            # discrete_features=False indicates continuous numerical data
            mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]
            mi_list.append(mi_val)

        # Convert to series for quantile calculations
        s_series = pd.Series(spearman_list).dropna()
        mi_series = pd.Series(mi_list).dropna()

        results.append({
            'feature': col,
            'med_spearman': s_series.median(),
            'q1_spearman': s_series.quantile(0.25),
            'q3_spearman': s_series.quantile(0.75),
            'med_mi': mi_series.median(),
            'q1_mi': mi_series.quantile(0.25),
            'q3_mi': mi_series.quantile(0.75)
        })

    # Return as a DataFrame sorted by Median Mutual Information
    return pd.DataFrame(results).set_index('feature').sort_values('med_mi', ascending=False)

# 2. Execution
stats_df = get_comprehensive_stats(oligo_data, features)

print("Comprehensive Feature Stats: Spearman vs Mutual Information")
print(stats_df)

In [ ]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression

# 1. Define the cohort column
oligo_data['cohort'] = oligo_data[CANONICAL_GENE] + "_" + oligo_data[CELL_LINE]

def get_stratified_stats(df, feature_cols, target_col=INHIBITION, mod_col=MODIFICATION):
    all_mod_results = []

    # Iterate through each chemical modification type
    for mod_type, mod_group in df.groupby(mod_col):
        mod_results = []

        for col in feature_cols:
            spearman_list = []
            mi_list = []

            # Group by cohort within this specific modification
            for cohort, cohort_group in mod_group.groupby('cohort'):
                valid_data = cohort_group[[col, target_col]].dropna()

                # Filter for statistical significance (minimum samples per cohort)
                if len(valid_data) < 7:
                    continue

                # Spearman
                s_val = valid_data[col].corr(valid_data[target_col], method='spearman')
                spearman_list.append(s_val)

                # Mutual Information
                x = valid_data[[col]]
                y = valid_data[target_col]
                mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]
                mi_list.append(mi_val)

            # Calculate stats if we have enough cohorts for this mod type
            if spearman_list:
                s_series = pd.Series(spearman_list).dropna()
                mi_series = pd.Series(mi_list).dropna()

                mod_results.append({
                    'modification': mod_type,
                    'feature': col,
                    'med_spearman': s_series.median(),
                    'q1_spearman': s_series.quantile(0.25),
                    'q3_spearman': s_series.quantile(0.75),
                    'med_mi': mi_series.median(),
                    'q1_mi': mi_series.quantile(0.25),
                    'q3_mi': mi_series.quantile(0.75),
                    'n_cohorts': len(s_series)
                })

        all_mod_results.extend(mod_results)

    # Create DataFrame and set hierarchical index
    result_df = pd.DataFrame(all_mod_results).set_index(['modification', 'feature'])
    return result_df.sort_index()

# 2. Execution
stratified_stats = get_stratified_stats(oligo_data, features)

# 3. Viewing the results
# You can now look at a specific modification, e.g., 'Gapmer'
print("Stats for specific modifications (Top 5 by MI):")
print(stratified_stats.sort_values(['modification', 'med_mi'], ascending=[True, False]))

In [ ]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# 1. Define the cohort column
oligo_data['cohort'] = oligo_data[CANONICAL_GENE] + "_" + oligo_data[CELL_LINE]
division_strategy = 'cohort'

# --- HELPER 1: Worker for Comprehensive Stats ---
def _process_comprehensive_feature(col, df, target_col):
    spearman_list, mi_list = [], []

    for cohort, group in df.groupby(division_strategy):
        valid_data = group[[col, target_col]].dropna()
        if len(valid_data) < 5:
            continue

        x, y = valid_data[[col]], valid_data[target_col]

        # Calculate Stats
        spearman_val = valid_data[col].corr(y, method='spearman')
        mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]

        spearman_list.append(spearman_val)
        mi_list.append(mi_val)

    if not spearman_list:
        return None  # Handle cases where a feature has completely invalid data

    s_series, mi_series = pd.Series(spearman_list).dropna(), pd.Series(mi_list).dropna()

    return {
        'feature': col,
        'med_spearman': s_series.median(),
        'q1_spearman': s_series.quantile(0.25),
        'q3_spearman': s_series.quantile(0.75),
        'med_mi': mi_series.median(),
        'q1_mi': mi_series.quantile(0.25),
        'q3_mi': mi_series.quantile(0.75)
    }


def get_comprehensive_stats_parallel(df, feature_cols, target_col=INHIBITION, n_jobs=-1):
    print(
        f"Calculating comprehensive stats for {len(feature_cols)} features on {n_jobs if n_jobs != -1 else 'all'} cores...")

    # Parallel execution
    results = Parallel(n_jobs=n_jobs)(
        delayed(_process_comprehensive_feature)(col, df, target_col)
        for col in tqdm(feature_cols, desc="Comprehensive")
    )

    # Clean out Nones and build DataFrame
    valid_results = [r for r in results if r is not None]
    return pd.DataFrame(valid_results).set_index('feature').sort_values('med_mi', ascending=False)


# --- HELPER 2: Worker for Stratified Stats ---
def _process_stratified_feature(col, df, target_col, mod_col):
    mod_results = []

    for mod_type, mod_group in df.groupby(mod_col):
        spearman_list, mi_list = [], []

        for cohort, cohort_group in mod_group.groupby(division_strategy):
            valid_data = cohort_group[[col, target_col]].dropna()
            if len(valid_data) < 7:
                continue

            x, y = valid_data[[col]], valid_data[target_col]

            spearman_val = valid_data[col].corr(y, method='spearman')
            mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]

            spearman_list.append(spearman_val)
            mi_list.append(mi_val)

        if spearman_list:
            s_series, mi_series = pd.Series(spearman_list).dropna(), pd.Series(mi_list).dropna()
            mod_results.append({
                'modification': mod_type,
                'feature': col,
                'med_spearman': s_series.median(),
                'q1_spearman': s_series.quantile(0.25),
                'q3_spearman': s_series.quantile(0.75),
                'med_mi': mi_series.median(),
                'q1_mi': mi_series.quantile(0.25),
                'q3_mi': mi_series.quantile(0.75),
                'n_cohorts': len(s_series)
            })

    return mod_results


def get_stratified_stats_parallel(df, feature_cols, target_col=INHIBITION, mod_col=MODIFICATION, n_jobs=-1):
    print(
        f"Calculating stratified stats for {len(feature_cols)} features on {n_jobs if n_jobs != -1 else 'all'} cores...")

    # Parallel execution
    results_nested = Parallel(n_jobs=n_jobs)(
        delayed(_process_stratified_feature)(col, df, target_col, mod_col)
        for col in tqdm(feature_cols, desc="Stratified")
    )

    # Flatten the nested lists returned by the workers
    all_mod_results = [item for sublist in results_nested for item in sublist]

    return pd.DataFrame(all_mod_results).set_index(['modification', 'feature']).sort_index()

In [ ]:
# --- MISSING LINK: Define competition_list and all_feature_cols ---
competition_list = [
    'PFRED_PLS', 'PFRED_SVM', 'OW_Overall', 'OW_Tm',
    'OW_Intra_Oligo', 'OW_Duplex', 'sfold_accessibility',
    'miranda_score', 'miranda_energy'
]

# Dynamically grab all numerical features, excluding IDs and target
exclude_cols = ['index_oligo', INHIBITION]
all_numeric_cols = oligo_data.select_dtypes(include=[np.number]).columns.tolist()
all_feature_cols = [col for col in all_numeric_cols if col not in exclude_cols]


In [ ]:
# Assuming you already defined 'all_feature_cols' and 'competition_list' from the previous step

# 1. Run Parallel Comprehensive
full_stats_df = get_comprehensive_stats_parallel(oligo_data, all_feature_cols, n_jobs=8)

# (Tagging code remains exactly the same...)
full_stats_df['feature_type'] = np.where(full_stats_df.index.isin(competition_list), 'Competition Tool', 'TAUSO Feature')
cols = ['feature_type', 'med_mi', 'med_spearman', 'q1_spearman', 'q3_spearman']
full_stats_df = full_stats_df[cols]

print(full_stats_df.head(15))

# 2. Run Parallel Stratified
stratified_stats_df = get_stratified_stats_parallel(oligo_data, all_feature_cols, n_jobs=8)
print(stratified_stats_df.sort_values(['modification', 'med_mi'], ascending=[True, False]).groupby(level=0).head(5))

In [ ]:
full_stats_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the visual style
sns.set_theme(style="whitegrid", palette="muted")

# 1. Prepare Data
# Reset indexes so we can use 'feature' as a column in seaborn
df_global = full_stats_df.reset_index()
df_strat = stratified_stats_df.reset_index()

# Set up the matplotlib figure layout (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('TAUSO vs Competition: Feature Performance Analysis', fontsize=20, fontweight='bold', y=0.98)

# ==========================================
# PLOT 1: The Performance Landscape (Scatter)
# ==========================================
# X-axis: Spearman (Directional correlation)
# Y-axis: Mutual Information (Total non-linear information)
ax1 = axes[0, 0]
sns.scatterplot(
    data=df_global, x='med_spearman', y='med_mi',
    hue='feature_type', style='feature_type',
    s=100, alpha=0.7, ax=ax1,
    palette={'TAUSO Feature': '#2ca02c', 'Competition Tool': '#d62728'}
)
ax1.set_title('Global Performance Landscape: Spearman vs Mutual Information', fontsize=14)
ax1.set_xlabel('Median Spearman Correlation')
ax1.set_ylabel('Median Mutual Information')
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)

# Label the top 5 TAUSO and top 3 Competition features so we know who is who
top_tauso = df_global[df_global['feature_type'] == 'TAUSO Feature'].nlargest(5, 'med_mi')
top_comp = df_global[df_global['feature_type'] == 'Competition Tool'].nlargest(3, 'med_mi')
for _, row in pd.concat([top_tauso, top_comp]).iterrows():
    ax1.text(row['med_spearman'] + 0.005, row['med_mi'], row['feature'], fontsize=9)

# ==========================================
# PLOT 2: The "Giga Chad" Leaderboard (Bar)
# ==========================================
# Top 20 features ranked by Mutual Information
ax2 = axes[0, 1]
top_20_global = df_global.nlargest(20, 'med_mi')
sns.barplot(
    data=top_20_global, x='med_mi', y='feature',
    hue='feature_type', ax=ax2, dodge=False,
    palette={'TAUSO Feature': '#2ca02c', 'Competition Tool': '#d62728'}
)
ax2.set_title('Top 20 Features Overall (by Mutual Information)', fontsize=14)
ax2.set_xlabel('Median Mutual Information')
ax2.set_ylabel('')

# ==========================================
# PLOT 3: Spearman Consistency Range (Error Bars)
# ==========================================
# Shows the median spearman and the Q1-Q3 spread for the top 15 features
ax3 = axes[1, 0]
top_15_sp = df_global.nlargest(15, 'med_mi')  # Take top by MI, but plot their Spearman spread
y_pos = np.arange(len(top_15_sp))

# Calculate error margins for matplotlib errorbar (distance from median)
xerr_lower = top_15_sp['med_spearman'] - top_15_sp['q1_spearman']
xerr_upper = top_15_sp['q3_spearman'] - top_15_sp['med_spearman']

ax3.errorbar(
    top_15_sp['med_spearman'], y_pos,
    xerr=[xerr_lower, xerr_upper], fmt='o', color='#1f77b4',
    ecolor='gray', elinewidth=2, capsize=4
)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(top_15_sp['feature'])
ax3.axvline(0, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Spearman Consistency (Median ± IQR) across Cohorts', fontsize=14)
ax3.set_xlabel('Spearman Correlation (Zero line in red)')
ax3.invert_yaxis()  # Highest MI at the top

# ==========================================
# PLOT 4: Modification / Chemistry Heatmap
# ==========================================
# How do the top 15 overall features perform across different chemistries?
ax4 = axes[1, 1]

# Filter for the top 15 global features
top_features_list = top_15_sp['feature'].tolist()
heat_data = df_strat[df_strat['feature'].isin(top_features_list)]

# Pivot to create a matrix: Rows = Features, Columns = Modifications, Values = MI
heat_matrix = heat_data.pivot(index='feature', columns='modification', values='med_mi')

# Reorder the rows to match the leaderboard ranking
heat_matrix = heat_matrix.loc[top_features_list]

# Plot the heatmap
sns.heatmap(heat_matrix, cmap='YlGnBu', ax=ax4, cbar_kws={'label': 'Median MI'}, annot=False)
ax4.set_title('Mutual Information by Chemical Modification', fontsize=14)
ax4.set_xlabel('Modification')
ax4.set_ylabel('')
# Rotate x-labels if they are long
plt.setp(ax4.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Final layout adjustments
plt.tight_layout()
plt.show()

In [ ]:
from tauso.data.consts import *
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# 1. Define the cohort column
oligo_data['cohort'] = oligo_data[CANONICAL_GENE] + "_" + oligo_data[CELL_LINE]
division_strategy = 'custom_id'


# --- HELPER 1: Worker for Comprehensive Stats ---
def _process_comprehensive_feature(col, df, target_col):
    spearman_list, mi_list = [], []

    for cohort, group in df.groupby(division_strategy):
        valid_data = group[[col, target_col]].dropna()
        if len(valid_data) < 5:
            continue

        x, y = valid_data[[col]], valid_data[target_col]

        # Calculate Stats
        spearman_val = valid_data[col].corr(y, method='spearman')
        mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]

        spearman_list.append(spearman_val)
        mi_list.append(mi_val)

    if not spearman_list:
        return None  # Handle cases where a feature has completely invalid data

    s_series, mi_series = pd.Series(spearman_list).dropna(), pd.Series(mi_list).dropna()

    return {
        'feature': col,
        'med_spearman': s_series.median(),
        'q1_spearman': s_series.quantile(0.25),
        'q3_spearman': s_series.quantile(0.75),
        'med_mi': mi_series.median(),
        'q1_mi': mi_series.quantile(0.25),
        'q3_mi': mi_series.quantile(0.75)
    }


def get_comprehensive_stats_parallel(df, feature_cols, target_col=INHIBITION, n_jobs=-1):
    print(
        f"Calculating comprehensive stats for {len(feature_cols)} features on {n_jobs if n_jobs != -1 else 'all'} cores...")

    # Parallel execution
    results = Parallel(n_jobs=n_jobs)(
        delayed(_process_comprehensive_feature)(col, df, target_col)
        for col in tqdm(feature_cols, desc="Comprehensive")
    )

    # Clean out Nones and build DataFrame
    valid_results = [r for r in results if r is not None]
    return pd.DataFrame(valid_results).set_index('feature').sort_values('med_mi', ascending=False)


# --- HELPER 2: Worker for Stratified Stats ---
def _process_stratified_feature(col, df, target_col, mod_col):
    mod_results = []

    for mod_type, mod_group in df.groupby(mod_col):
        spearman_list, mi_list = [], []

        for cohort, cohort_group in mod_group.groupby(division_strategy):
            valid_data = cohort_group[[col, target_col]].dropna()
            if len(valid_data) < 7:
                continue

            x, y = valid_data[[col]], valid_data[target_col]

            spearman_val = valid_data[col].corr(y, method='spearman')
            mi_val = mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]

            spearman_list.append(spearman_val)
            mi_list.append(mi_val)

        if spearman_list:
            s_series, mi_series = pd.Series(spearman_list).dropna(), pd.Series(mi_list).dropna()
            mod_results.append({
                'modification': mod_type,
                'feature': col,
                'med_spearman': s_series.median(),
                'q1_spearman': s_series.quantile(0.25),
                'q3_spearman': s_series.quantile(0.75),
                'med_mi': mi_series.median(),
                'q1_mi': mi_series.quantile(0.25),
                'q3_mi': mi_series.quantile(0.75),
                'n_cohorts': len(s_series)
            })

    return mod_results


def get_stratified_stats_parallel(df, feature_cols, target_col=INHIBITION, mod_col=MODIFICATION, n_jobs=-1):
    print(
        f"Calculating stratified stats for {len(feature_cols)} features on {n_jobs if n_jobs != -1 else 'all'} cores...")

    # Parallel execution
    results_nested = Parallel(n_jobs=n_jobs)(
        delayed(_process_stratified_feature)(col, df, target_col, mod_col)
        for col in tqdm(feature_cols, desc="Stratified")
    )

    # Flatten the nested lists returned by the workers
    all_mod_results = [item for sublist in results_nested for item in sublist]

    return pd.DataFrame(all_mod_results).set_index(['modification', 'feature']).sort_index()


In [ ]:
competition_list = [
    'PFRED_PLS', 'PFRED_SVM', 'OW_Overall', 'OW_Tm',
    'OW_Intra_Oligo', 'OW_Duplex', 'sfold_accessibility',
    'miranda_score', 'miranda_energy'
]

# Dynamically grab all numerical features, excluding IDs and target
exclude_cols = ['index_oligo', INHIBITION]
all_numeric_cols = oligo_data.select_dtypes(include=[np.number]).columns.tolist()
all_feature_cols = [col for col in all_numeric_cols if col not in exclude_cols]


In [ ]:
# 1. Run Parallel Comprehensive
full_stats_df = get_comprehensive_stats_parallel(oligo_data, all_feature_cols, n_jobs=8)

# (Tagging code remains exactly the same...)
full_stats_df['feature_type'] = np.where(full_stats_df.index.isin(competition_list), 'Competition Tool',
                                         'TAUSO Feature')
cols = ['feature_type', 'med_mi', 'med_spearman', 'q1_spearman', 'q3_spearman']
full_stats_df = full_stats_df[cols]

print(full_stats_df.head(15))

# 2. Run Parallel Stratified
stratified_stats_df = get_stratified_stats_parallel(oligo_data, all_feature_cols, n_jobs=8)
print(stratified_stats_df.sort_values(['modification', 'med_mi'], ascending=[True, False]).groupby(level=0).head(5))
full_stats_df

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set the visual style
sns.set_theme(style="whitegrid", palette="muted")

# 1. Prepare Data
# Reset indexes so we can use 'feature' as a column in seaborn
df_global = full_stats_df.reset_index()
df_strat = stratified_stats_df.reset_index()

# Set up the matplotlib figure layout (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('TAUSO vs Competition: Feature Performance Analysis', fontsize=20, fontweight='bold', y=0.98)

# ==========================================
# PLOT 1: The Performance Landscape (Scatter)
# ==========================================
# X-axis: Spearman (Directional correlation)
# Y-axis: Mutual Information (Total non-linear information)
ax1 = axes[0, 0]
sns.scatterplot(
    data=df_global, x='med_spearman', y='med_mi',
    hue='feature_type', style='feature_type',
    s=100, alpha=0.7, ax=ax1,
    palette={'TAUSO Feature': '#2ca02c', 'Competition Tool': '#d62728'}
)
ax1.set_title('Global Performance Landscape: Spearman vs Mutual Information', fontsize=14)
ax1.set_xlabel('Median Spearman Correlation')
ax1.set_ylabel('Median Mutual Information')
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)

# Label the top 5 TAUSO and top 3 Competition features so we know who is who
top_tauso = df_global[df_global['feature_type'] == 'TAUSO Feature'].nlargest(5, 'med_mi')
top_comp = df_global[df_global['feature_type'] == 'Competition Tool'].nlargest(3, 'med_mi')
for _, row in pd.concat([top_tauso, top_comp]).iterrows():
    ax1.text(row['med_spearman'] + 0.005, row['med_mi'], row['feature'], fontsize=9)

# ==========================================
# PLOT 2: The "Giga Chad" Leaderboard (Bar)
# ==========================================
# Top 20 features ranked by Mutual Information
ax2 = axes[0, 1]
top_20_global = df_global.nlargest(20, 'med_mi')
sns.barplot(
    data=top_20_global, x='med_mi', y='feature',
    hue='feature_type', ax=ax2, dodge=False,
    palette={'TAUSO Feature': '#2ca02c', 'Competition Tool': '#d62728'}
)
ax2.set_title('Top 20 Features Overall (by Mutual Information)', fontsize=14)
ax2.set_xlabel('Median Mutual Information')
ax2.set_ylabel('')

# ==========================================
# PLOT 3: Spearman Consistency Range (Error Bars)
# ==========================================
# Shows the median spearman and the Q1-Q3 spread for the top 15 features
ax3 = axes[1, 0]
top_15_sp = df_global.nlargest(15, 'med_mi')  # Take top by MI, but plot their Spearman spread
y_pos = np.arange(len(top_15_sp))

# Calculate error margins for matplotlib errorbar (distance from median)
xerr_lower = top_15_sp['med_spearman'] - top_15_sp['q1_spearman']
xerr_upper = top_15_sp['q3_spearman'] - top_15_sp['med_spearman']

ax3.errorbar(
    top_15_sp['med_spearman'], y_pos,
    xerr=[xerr_lower, xerr_upper], fmt='o', color='#1f77b4',
    ecolor='gray', elinewidth=2, capsize=4
)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(top_15_sp['feature'])
ax3.axvline(0, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Spearman Consistency (Median ± IQR) across Cohorts', fontsize=14)
ax3.set_xlabel('Spearman Correlation (Zero line in red)')
ax3.invert_yaxis()  # Highest MI at the top

# ==========================================
# PLOT 4: Modification / Chemistry Heatmap
# ==========================================
# How do the top 15 overall features perform across different chemistries?
ax4 = axes[1, 1]

# Filter for the top 15 global features
top_features_list = top_15_sp['feature'].tolist()
heat_data = df_strat[df_strat['feature'].isin(top_features_list)]

# Pivot to create a matrix: Rows = Features, Columns = Modifications, Values = MI
heat_matrix = heat_data.pivot(index='feature', columns='modification', values='med_mi')

# Reorder the rows to match the leaderboard ranking
heat_matrix = heat_matrix.loc[top_features_list]

# Plot the heatmap
sns.heatmap(heat_matrix, cmap='YlGnBu', ax=ax4, cbar_kws={'label': 'Median MI'}, annot=False)
ax4.set_title('Mutual Information by Chemical Modification', fontsize=14)
ax4.set_xlabel('Modification')
ax4.set_ylabel('')
# Rotate x-labels if they are long
plt.setp(ax4.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Final layout adjustments
plt.tight_layout()
plt.show()